# Treino do córtex v3 `medio` (Colab) — RoPE + RMSNorm + SwiGLU + BPE + agenda WSD

Modelo: `medio` (12 camadas, 12 cabeças, d=768, contexto 512). Com vocabulário BPE de 16 000 tokens tem **≈97M parâmetros** (≈85M nos blocos + ≈12M no embedding amarrado).
Corpus: `textos/corpus_portugues.txt` (treino) e `textos/corpus_validacao.txt` (validação separada, sem vazamento).

**Antes:** *Ambiente de execução → Alterar tipo de ambiente de execução → GPU* (T4 serve; A100 é ~8x mais rápida).
**No Drive:** coloque `apolo_treino_v3.zip` em `MyDrive/apolo/`. Ele deve conter: `cortex_v3.py`, `tokenizador_bpe.py`, `textos/corpus_portugues.txt`, `textos/corpus_validacao.txt`.
Tudo é salvo em `MyDrive/apolo/cerebro_v3_medio/` (pasta nova; o `cerebro_v2` não é tocado). Checkpoint a cada 2000 passos, gravado de forma atômica (escreve em `.tmp` e renomeia). Se a sessão cair: rode as células 1 a 3 e a célula **Retomar** — a retomada é exata (pesos, otimizador, escala AMP, passo e sorteios de dados).

## Estimativa honesta de tempo e custo (é só estimativa!)
Regra: FLOPs de treino ≈ 6 · parâmetros · tokens (+~10% da atenção). Com 97M parâmetros, cada token custa ≈ 0,64 GFLOP. Tokens por passo = lote × acumular × contexto = 16 × 4 × 512 = **32 768**.

| | T4 (fp16) | A100 (bf16) |
|---|---|---|
| Vazão sustentada que assumo | ~8–12 TFLOP/s (pico 65, mas ela esquenta e o PyTorch sem `compile` rende ~15–20%) | ~80–120 TFLOP/s (pico 312, ~30–40%) |
| Tokens/s resultante | ~12–19 mil | ~125–190 mil |
| **4 voltas no corpus (~215M tokens, ~6 600 passos)** | **~3–5 horas** | **~20–40 minutos** |
| Unidades de computação do Colab (aprox.) | ~5–9 (T4 ≈ 1,8/h) | ~4–8 (A100 ≈ 8–13/h) |

As unidades por hora mudam; confira o seu plano. A estimativa de ~54M tokens no corpus (187 MB ÷ ~3,5 caracteres/token) será conferida na célula 4, que recalcula `PASSOS` com o número real.

**Meta de ~20 tokens por parâmetro (≈1,9 bilhão de tokens): NÃO é viável com este corpus.** Seriam ~36 voltas sobre ~54M tokens: o modelo decoraria o texto (sobreajuste), e na T4 levaria ~35 horas. O viável: **~4 voltas (~2 tokens/parâmetro)**, que a literatura de dados repetidos mostra ser quase tão bom quanto dados novos; acima disso, o ganho só vem de MAIS dados. Se o dono juntar ≥500M tokens de texto, a A100 faria 20 tokens/parâmetro em ~4 horas. Mude `EPOCAS` na célula 4 conforme o tempo disponível; o platô WSD permite estender depois (`--estender`).

## 1. GPU e Google Drive

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA (ative a GPU antes de continuar)")
from google.colab import drive
drive.mount("/content/drive")

## 2. Descompactar o pacote

In [ ]:
import os
ZIP = "/content/drive/MyDrive/apolo/apolo_treino_v3.zip"
assert os.path.exists(ZIP), f"Não achei {ZIP}. Coloque o zip no Drive, em MyDrive/apolo/."
!rm -rf /content/apolo && mkdir -p /content/apolo && unzip -q -o "{ZIP}" -d /content/apolo
%cd /content/apolo
!ls -la . textos | head -15
!pip -q install tokenizers==0.21.1

## 3. Treinar o tokenizador BPE (vocabulário 16 000, nível de byte)
Vocabulário maior comprime mais (mais texto por token), mas gasta parâmetros de embedding e deixa palavras raras pouco treinadas; 16k é o meio-termo (embedding ≈12M no `medio`). O nível de byte garante que nenhum caractere (nem `[ ] { } < > \ | `` ` ``) seja desconhecido. Demora alguns minutos e é salvo no Drive junto do checkpoint; se já existir, não refaz.

In [ ]:
PASTA = "/content/drive/MyDrive/apolo/cerebro_v3_medio"
os.makedirs(PASTA, exist_ok=True)
TOK = f"{PASTA}/tokenizador.json"
if os.path.exists(TOK):
    print("tokenizador já existe no Drive; reaproveitando:", TOK)
else:
    !python -u tokenizador_bpe.py treinar --dados textos/corpus_portugues.txt --saida "{TOK}" --vocab 16000
!python tokenizador_bpe.py testar --tokenizador "{TOK}" --dados textos/corpus_validacao.txt

## 4. Configurar o treino
- `LOTE × ACUMULAR × 512` = tokens por passo (32 768 com 16×4). Na A100 pode usar `LOTE = 32, ACUMULAR = 2` (mesmos tokens por passo, mais rápido). Se faltar memória na T4, baixe `LOTE` para 8 e suba `ACUMULAR` para 8.
- `EPOCAS` = quantas voltas no corpus (4 é o recomendado aqui; veja a estimativa no topo).
- Agenda WSD: aquece 300 passos, mantém `LR` constante até 80% e desce linearmente nos 20% finais.

In [ ]:
import math
from tokenizador_bpe import carregar_tokenizador, pedacos_de_texto
tk = carregar_tokenizador(TOK)
amostra = "".join(pedacos_de_texto("textos/corpus_portugues.txt", 20))
car_por_token = len(amostra) / len(tk.codificar(amostra))
n_car = os.path.getsize("textos/corpus_portugues.txt")  # bytes ~ caracteres (aproximação)
n_tokens = n_car / car_por_token

TAMANHO = "medio"
LOTE, ACUMULAR, LR, EPOCAS = 16, 4, 6e-4, 4
tokens_passo = LOTE * ACUMULAR * 512
PASSOS = math.ceil(EPOCAS * n_tokens / tokens_passo)
print(f"{car_por_token:.2f} caracteres/token | ~{n_tokens/1e6:.0f}M tokens no corpus | {tokens_passo:,} tokens/passo")
print(f"PASSOS = {PASSOS} ({EPOCAS} voltas, ~{PASSOS*tokens_passo/97.3e6:.1f} tokens/parâmetro)")

## 5. Treinar
O log mostra tokens/s (use para conferir a estimativa), perda de treino e de validação e **BPC** (bits por caractere; quanto menor, melhor; é a medida que compara com o modelo de caractere do v2).

In [ ]:
!python -u -W ignore cortex_v3.py treinar --dados textos/corpus_portugues.txt --val_dados textos/corpus_validacao.txt \
  --tamanho {TAMANHO} --tokenizador bpe --tokenizador_json "{TOK}" --passos {PASSOS} --lote {LOTE} --acumular {ACUMULAR} \
  --lr {LR} --aquecimento 300 --descida 0.2 --avaliar_cada 500 --log_cada 50 --salvar_cada 2000 --pasta "{PASTA}"

### (Só se a sessão caiu) Retomar
Continua EXATAMENTE do último `checkpoint.pt` (rode antes as células 1 a 4, para as variáveis existirem). A agenda de taxa, o lote e as sementes vêm do checkpoint; só os dados e a pasta vêm daqui.

In [ ]:
!python -u -W ignore cortex_v3.py treinar --dados textos/corpus_portugues.txt --val_dados textos/corpus_validacao.txt \
  --tokenizador bpe --pasta "{PASTA}" --retomar --avaliar_cada 500 --log_cada 50 --salvar_cada 2000

### (Opcional) Treinar mais depois de terminar
Para continuar o platô em vez de recomeçar, use `--retomar --estender` com um `--passos` maior. O ideal é ter parado no platô (um checkpoint de antes da descida); se o treino já desceu a taxa até zero, ela volta ao platô (aviso no log). Exemplo: `--passos 10000 --estavel_ate 0.8`.

## 6. Bits por caractere na validação e teste com prompts
O Apolo manda a pergunta seguida de uma quebra de linha; o córtex escreve a continuação e para em `\n\n`.

In [ ]:
!python -W ignore cortex_v3.py avaliar --val_dados textos/corpus_validacao.txt --pasta "{PASTA}"

In [ ]:
import cortex_v3
modelo, tk, dev = cortex_v3.carregar(PASTA)
perguntas = ["Oi", "Oi, tudo bem?", "Qual é o seu nome?", "O que é um substantivo?", "Qual é a capital de Rondônia?",
             "Como mostrar uma mensagem na tela em Python?", "Como criar uma lista em Python?",
             "Como escrever um laço for em Python?", "Como filtrar linhas com WHERE em SQL?",
             "Como criar uma branch no Git?", "O que é um array em JavaScript?", "Como listar os containers do Docker?"]
for p in perguntas:
    print(f"> {p}\n{cortex_v3.continuar(modelo, tk, dev, p + chr(10), 200, temperatura=0.5, parar_em=[chr(10) * 2])}\n")

## 7. Baixar os arquivos
Baixe `modelo.pt` (leve, só para servir) e o `tokenizador.json`; coloque em `apolo/cerebro_v3_medio/` e avise o Claude Code. O `checkpoint.pt` completo (com otimizador, ~1,2 GB) só serve para continuar o treino e fica no Drive.

In [ ]:
from google.colab import files
files.download(f"{PASTA}/modelo.pt")
files.download(f"{PASTA}/tokenizador.json")